Assignmnent:


In [1]:
import pandas as pd
import numpy as np
import sys
import plotly.express as px
import plotly.graph_objects as go
from time import time
from tqdm import tqdm
from itertools import product

sys.path.append('../../')
from src.lib.signals.cusum_filter import OnlineCUSUMFilter

In [2]:
PERIOD = 5    # minutes
DAILY_PERIODS = 6.5 * 60 / PERIOD    # number of trading periods in a day
ANNUAL_FACTOR = 252 * DAILY_PERIODS
STOCK = 'SPY'

In [3]:
# load OHLCV data from market_data/intraday/
ohlcv = pd.read_csv(f'../../market_data/intraday/{STOCK}_{PERIOD}mn.csv')
ohlcv = ohlcv.set_index('Date')
ohlcv.index = pd.to_datetime(ohlcv.index)
ohlcv.tail()


,Open,High,Low,Close,Volume
Date,,,,,
2026-03-17 19:35:00+00:00,671.00,671.14,670.94,670.99,11439.0
2026-03-17 19:40:00+00:00,670.99,671.15,670.75,670.86,18665.0
2026-03-17 19:45:00+00:00,670.87,671.21,670.85,671.18,23100.0
2026-03-17 19:50:00+00:00,671.18,671.28,670.58,671.21,44420.0
2026-03-17 19:55:00+00:00,671.20,671.20,670.47,670.73,73989.0


In [4]:
log_px = np.log(ohlcv['Close'])
# calculate returns
returns = ohlcv['Close'].pct_change()

# backtest period
nb_years = (ohlcv.index[-1] - ohlcv.index[0]).days / 365

In [5]:
# CUSUM hyperparameters
K_RANGE = np.arange(0, 0.25, 0.05)    # allowance for zscore (to prevent false positives)
H_BASE_RANGE = np.arange(2, 5.5, 0.5)    # base threshold for CUSUM alarm
SPAN_RANGE = np.arange(20, 55, 5)    # number of periods to calculate the moving average
MA_PERIOD_RANGE = np.arange(100, 405, 50)    # moving average period

In [6]:
# ── Strategy integration ─────────────────────────────────────────────
def apply_trend_filter(prices, signals, ma_period):
    """Enter long/short trend on CUSUM alarm + MA confirmation."""
    mov_avg  = prices.rolling(ma_period).mean()
    long     = (signals == +1) & (prices > mov_avg)
    short    = (signals == -1) & (prices < mov_avg)
    return long.astype(int) - short.astype(int)

In [7]:
# run grid search on hyperparameters and optimize for Sharpe ratio
best_params = {}
best_sharpe = 0
best_returns = None
best_signals = None

outputs = []
for k, h_base, span, ma_period in tqdm(product(K_RANGE, H_BASE_RANGE, SPAN_RANGE, MA_PERIOD_RANGE)):
    output = {'k': k, 'h_base': h_base, 'span': span, 'ma_period': ma_period}

    # ── CUSUM filter ─────────────────────────────────────────────
    detector = OnlineCUSUMFilter(k=k, h_base=h_base, span=span)
    signals  = detector.run(log_px)

    # ── Count breakouts ──────────────────────────────────────────
    up   = signals[signals == +1]
    down = signals[signals == -1]
    output['up_breaks'] = len(up)
    output['down_breaks'] = len(down)

    # ── Apply trend filter (combine with moving average) ────────
    trend_signal = apply_trend_filter(ohlcv['Close'], signals, ma_period)

    # use trend signal as positions (ie: fixed trade size)
    position = trend_signal.copy()

    # calculate strategy returns (position is lagged by 1 to avoid look-ahead bias)
    strategy_returns = position.shift(1) * returns

    # calculate cumulative returns
    cumulative_return = (1 + strategy_returns).cumprod()

    output['total_return'] = cumulative_return.iloc[-1]

    # calculate risk-adjusted performance metrics
    annualized_return = (cumulative_return.iloc[-1]-1) / nb_years
    annualized_volatility = cumulative_return.pct_change().std() * np.sqrt(ANNUAL_FACTOR)
    if annualized_volatility == 0:
        continue
    sharpe_ratio = annualized_return / annualized_volatility
    max_drawdown = (cumulative_return.cummax()-cumulative_return).max()

    output['annual_return'] = annualized_return
    output['annual_vol'] = annualized_volatility
    output['sharpe'] = sharpe_ratio
    output['max_drawdown'] = max_drawdown
    outputs.append(output)

    if sharpe_ratio > 1 and sharpe_ratio > best_sharpe:
        best_sharpe = sharpe_ratio
        best_params = output
        best_returns = cumulative_return
        best_signals = signals

        print(f"Best parameters: {best_params}")

185it [01:11,  2.57it/s]

Best parameters: {'k': np.float64(0.0), 'h_base': np.float64(3.5), 'span': np.int64(45), 'ma_period': np.int64(200), 'up_breaks': 4541, 'down_breaks': 3244, 'total_return': np.float64(1.1363484446882721), 'annual_return': np.float64(0.06826773979591129), 'annual_vol': np.float64(0.06655794683974442), 'sharpe': np.float64(1.025688787550548), 'max_drawdown': np.float64(0.054431225154231155)}


186it [01:12,  2.55it/s]

Best parameters: {'k': np.float64(0.0), 'h_base': np.float64(3.5), 'span': np.int64(45), 'ma_period': np.int64(250), 'up_breaks': 4541, 'down_breaks': 3244, 'total_return': np.float64(1.1413241605343436), 'annual_return': np.float64(0.07075901041842993), 'annual_vol': np.float64(0.06541563383127573), 'sharpe': np.float64(1.0816834795323726), 'max_drawdown': np.float64(0.04854066809389823)}


213it [01:22,  2.55it/s]

...

800it [05:15,  2.50it/s]

Best parameters: {'k': np.float64(0.1), 'h_base': np.float64(3.0), 'span': np.int64(30), 'ma_period': np.int64(150), 'up_breaks': 4658, 'down_breaks': 3419, 'total_return': np.float64(1.2603365572978753), 'annual_return': np.float64(0.1303468359584698), 'annual_vol': np.float64(0.06853628619565495), 'sharpe': np.float64(1.9018660507276435), 'max_drawdown': np.float64(0.06441793865904133)}


801it [05:16,  2.53it/s]

Best parameters: {'k': np.float64(0.1), 'h_base': np.float64(3.0), 'span': np.int64(30), 'ma_period': np.int64(200), 'up_breaks': 4658, 'down_breaks': 3419, 'total_return': np.float64(1.2925714418754248), 'annual_return': np.float64(0.14648638722157759), 'annual_vol': np.float64(0.06707497163145766), 'sharpe': np.float64(2.1839202262573387), 'max_drawdown': np.float64(0.05414682697088746)}


1715it [11:23,  2.51it/s]


In [11]:
# plot ohlcv['Close'] and best_returns on same chart but 2 separate y axes
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=ohlcv.index,
        y=ohlcv["Close"],
        mode="lines",
        name=f"{STOCK} Price",
        yaxis="y",
    )
)
fig.add_trace(
    go.Scatter(
        x=best_returns.index,
        y=best_returns - 1,
        mode="lines",
        name="Breakout Strategy",
        yaxis="y2",
    )
)
fig.update_layout(
    xaxis=dict(title=None),
    yaxis=dict(title="Price", side="left", showgrid=True),
    yaxis2=dict(
        title="Cumulative return",
        overlaying="y",
        side="right",
        tickformat=".0%",
        showgrid=False,
    ),
    legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top"),
)
fig.show()

In [12]:
# display outputs and short by Sharpe ratio
outputs_df = pd.DataFrame(outputs)
outputs_df = outputs_df.sort_values(by='sharpe', ascending=False)
outputs_df


,k,h_base,span,ma_period,up_breaks,down_breaks,total_return,annual_return,annual_vol,sharpe,max_drawdown
800,0.10,3.0,30,200,4658,3419,1.292571,0.146486,0.067075,2.183920,0.054147
1573,0.20,4.0,20,350,3152,2362,1.230246,0.115281,0.053475,2.155813,0.035908
801,0.10,3.0,30,250,4658,3419,1.284141,0.142266,0.066733,2.131876,0.044563
1574,0.20,4.0,20,400,3152,2362,1.216173,0.108235,0.053550,2.021192,0.042686
1572,0.20,4.0,20,300,3152,2362,1.202796,0.101537,0.053278,1.905790,0.032678
...,...,...,...,...,...,...,...,...,...,...,...
389,0.05,2.0,50,300,6842,4899,0.817088,-0.091582,0.082402,-1.111407,0.200664
390,0.05,2.0,50,350,6842,4899,0.818406,-0.090921,0.080680,-1.126936,0.191622
1026,0.10,5.0,50,300,2817,1994,0.883110,-0.058525,0.051673,-1.132607,0.137204
104,0.00,3.0,20,400,5530,4156,0.837607,-0.081308,0.067249,-1.209054,0.162393


In [ ]:
# TODO: investigate hyperparameter stability (dSharpe/dParams, cross-validation)
# TODO: add transaction costs
# TODO: identify market regimes where the strategy performs well/poorly
# TODO: augment with other signals (VIX?)